# StrikeVision — Pose Estimation

Pipeline general para cualquier round: `PersonDetector@production` + ByteTrack + MediaPipe Pose Landmarker. La salida conserva los IDs de cada fragmento de track y genera keypoints en coordenadas del frame original.

In [1]:
from pathlib import Path
import json
import pandas as pd
import yaml

from ufc_tracker.detection.weights import project_root
from ufc_tracker.pose.pipeline import run_pose_pipeline

ROOT = project_root(Path.cwd())
CONFIG_PATH = ROOT / 'configs/app/pose_pipeline.yaml'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
config

{'output_root': 'data/processed/poses',
 'tracking_confidence': 0.5,
 'min_track_frames': 15,
 'max_frames': None}

## Seleccionar un round

La tabla usa el manifiesto versionado del dataset. Cambia `VIDEO_RELATIVE_PATH` por cualquier fila antes de ejecutar el pipeline.

In [2]:
manifest = pd.read_csv(ROOT / 'data/metadata/splits_manifest.csv')
display(manifest[['category', 'fight_id', 'round_number', 'relative_path']])
VIDEO_RELATIVE_PATH = manifest.iloc[0]['relative_path']
VIDEO_PATH = ROOT / VIDEO_RELATIVE_PATH
OUTPUT_DIR = ROOT / config['output_root'] / VIDEO_PATH.stem
VIDEO_PATH, OUTPUT_DIR

,category,fight_id,round_number,relative_path
0,aggressive_men,adesanya_pereira_1,1,data/splits/aggressive_men/adesanya_pereira_1_...
1,aggressive_men,adesanya_pereira_1,2,data/splits/aggressive_men/adesanya_pereira_1_...
2,aggressive_men,adesanya_pereira_1,3,data/splits/aggressive_men/adesanya_pereira_1_...
3,aggressive_men,adesanya_pereira_1,4,data/splits/aggressive_men/adesanya_pereira_1_...
4,aggressive_men,adesanya_pereira_1,5,data/splits/aggressive_men/adesanya_pereira_1_...
5,aggressive_men,holloway_gaethje,1,data/splits/aggressive_men/holloway_gaethje__h...
6,aggressive_men,holloway_gaethje,2,data/splits/aggressive_men/holloway_gaethje__h...
7,aggressive_men,holloway_gaethje,3,data/splits/aggressive_men/holloway_gaethje__h...
8,aggressive_men,holloway_gaethje,4,data/splits/aggressive_men/holloway_gaethje__h...
9,aggressive_men,holloway_gaethje,5,data/splits/aggressive_men/holloway_gaethje__h...


(WindowsPath('C:/Users/Cristian/Documents/UFC TRACKER/data/splits/aggressive_men/adesanya_pereira_1__israel_adesanya_vs_alex_pereira_1__aggressive_men_round1.mp4'),
 WindowsPath('C:/Users/Cristian/Documents/UFC TRACKER/data/processed/poses/adesanya_pereira_1__israel_adesanya_vs_alex_pereira_1__aggressive_men_round1'))

## Ejecutar MediaPipe Pose

Usa `max_frames=300` para desarrollo. Cambia a `None` únicamente después de revisar el preview corto.

In [8]:
# result = run_pose_pipeline(
#     VIDEO_PATH,
#     OUTPUT_DIR,
#     tracking_confidence=float(config['tracking_confidence']),
#     min_track_frames=int(config['min_track_frames']),
#     max_frames=300,
# )
# #result

In [ ]:
# metrics = json.loads(result.metrics_path.read_text(encoding='utf-8'))
# metrics['metrics']['overall']

## Criterios de revisión

Abre `result.preview_path` y revisa muñecas, codos, rodillas y tobillos durante guardia, ataque y oclusiones. `pose_coverage` mide poses con al menos cuatro keypoints esenciales; la disponibilidad por keypoint muestra qué articulaciones se pierden.

## Resumen del notebook (pose estimation)

### Qué se hizo (visión general)

Pipeline de **pose estimation sobre los dos peleadores** de un round, encadenado al detector ya versionado: `PersonDetector@production` + ByteTrack entregan las cajas, y **MediaPipe Pose Landmarker** extrae 17 keypoints por peleador en coordenadas del frame original.

Flujo en capas:

1. **Tracking de peleadores** — `extract_fighter_tracking()` reutiliza `track_video()` + `select_fighter_tracks()` del módulo de detección y exige exactamente **2 peleadores** por frame.
2. **Pose por crop** — cada bbox se recorta y se pasa a MediaPipe; los keypoints se devuelven al sistema de coordenadas del frame completo.
3. **Artefactos reproducibles** — `tracking.jsonl`, `pose.jsonl`, `pose_preview.mp4`, `pose_metrics.json` y `run_metadata.json` en `data/processed/poses/<round>/`.

---

### Funciones importantes

| Función | Rol |
|---------|-----|
| `run_pose_pipeline()` | Orquesta tracking → pose → preview → métricas → metadata |
| `extract_fighter_tracking()` | ByteTrack vía `PersonDetector`; exporta `TrackingRecord` por frame |
| `resolve_mediapipe_pose_model()` | Descarga `pose_landmarker_lite.task` en `models/weights/` |
| `MediaPipePoseEstimator.estimate()` | Pose sobre el crop; un task `RunningMode.VIDEO` por `fighter_id` |
| `estimate_pose_records()` | Recorre frames, recorta bbox y arma cada `PoseRecord` |
| `_pose_is_valid()` | Marca `pose_valid` con ≥ 4 de 10 keypoints esenciales |
| `calculate_metrics()` | Cobertura, disponibilidad, continuidad temporal y latencia |
| `render_pose_preview()` | Dibuja esqueleto, bbox y `fighter_id` (azul/rojo) |
| `_mark_missing()` | Resetea el estado temporal cuando un peleador desaparece |

---

### Problemas encontrados

- **MediaPipe es single-person** — el Pose Landmarker devuelve una sola pose por imagen; con dos peleadores en el frame hay que recortar por bbox y correr el modelo por separado.
- **Estado temporal contaminado** — en `RunningMode.VIDEO` el backend suaviza entre frames; si un track desaparece y vuelve, arrastra la pose anterior del otro cuerpo.
- **Coordenadas relativas al crop** — MediaPipe entrega landmarks normalizados al recorte, no al frame; sin reescalar y sumar el offset, el esqueleto queda desplazado.
- **Fragmentación de tracks heredada** — sigue viniendo de la capa de detección; por eso el `fighter_id` es `fighter_track_<tid>` y no `fighter_red` / `fighter_blue`.
- **`min_track_frames` fijo** — con clips cortos o `max_frames` bajo, un umbral de 15 frames descartaba tracks válidos y el pipeline fallaba por “menos de dos peleadores”.
- **Confidence perdida** — `track_video()` devolvía tuplas `(box, poly)` y el score se descartaba; sin él no se puede filtrar ni auditar después.
- **Oclusiones y clinch** — muñecas y tobillos son los keypoints que más se pierden; son justo los que necesita la clasificación de golpes.
- **Desfase tracking vs video** — si el número de frames leídos no coincide con el esperado, los índices dejan de alinearse y los artefactos quedan corruptos en silencio.

---

### Métodos de solución importantes

1. **Contratos compartidos** — `TrackObservation` y `TrackingRecord` (`ufc_tracker.tracking.contracts`) fijan un formato serializable entre detección, pose y las etapas temporales futuras; ahí viaja también la `confidence`.
2. **Crop + offset** — `_with_offset()` devuelve todos los keypoints al frame original, así el preview y las features posteriores comparten un solo sistema de coordenadas.
3. **Un modelo por peleador + `mark_missing()`** — cada `fighter_id` tiene su propio `PoseLandmarker`, y al perder visibilidad se cierra para no contaminar el suavizado.
4. **Validez explícita** — `REQUIRED_KEYPOINTS` (10 articulaciones) con `MIN_VALID_REQUIRED_KEYPOINTS = 4`; una pose parcial se marca `pose_valid=False` en vez de descartarse.
5. **Frames no visibles registrados** — se escriben filas con `visible=False` en lugar de omitirlas, para que el eje temporal quede completo.
6. **Umbral adaptativo** — `min(min_track_frames, ceil(5% de frames))` evita que clips cortos rompan la selección de peleadores.
7. **Fallo temprano** — el pipeline lanza error si hay menos de dos peleadores, más de dos en un frame, o si el conteo de frames no cuadra.
8. **Metadata de reproducibilidad** — `run_metadata.json` guarda commit de git, versiones de `ultralytics` / `mediapipe` / OpenCV y la configuración efectiva.

---

### Parámetros clave (ajuste rápido)

| Parámetro | Uso |
|-----------|-----|
| `tracking_confidence` | Umbral de detección antes de ByteTrack (`configs/app/pose_pipeline.yaml`) |
| `min_track_frames` | Frames mínimos por track; el efectivo es `min(config, 5% del total)` |
| `max_frames` | Frames a analizar; `300` en desarrollo, `None` solo tras revisar el preview |
| `min_visibility` | Visibilidad mínima por keypoint en MediaPipe (default `0.5`) |
| `MIN_VALID_REQUIRED_KEYPOINTS` | Cuántas articulaciones esenciales exigir para `pose_valid` |
| `output_root` | Carpeta de artefactos, por defecto `data/processed/poses` |

---

### Nota para el desarrollador — próxima mejora

**Falta el Model Registry y la ruta de producción.** El detector ya carga desde `models:/PersonDetector@production`, pero MediaPipe todavía se resuelve por descarga directa a `models/weights/`. Pendientes:

- **Registrar `PoseEstimator` en MLflow** — replicar el patrón de `ufc_tracker/ml/registry.py`: descripción en inglés, inputs (`video_path`, `frames`), alias `@production` y un `resolve_pose_estimator_weight()` con fallback local.
- **Función de producción** — equivalente a `send_prediction()` para exponer el pipeline de pose desde la API y el laboratorio web.
- **Merge de fragmentos de track** — sigue siendo el cuello de botella heredado; hasta resolverlo, `fighter_id` no puede mapearse a `fighter_red` / `fighter_blue`.
- **Comparar backends** — el `Protocol PoseEstimator` ya permite enchufar MoveNet, YOLO-pose o RTMPose y medirlos con las mismas métricas.
- **Suavizado e interpolación** — rellenar huecos cortos de muñecas y tobillos para no romper las ventanas temporales de golpe.
- **Features derivadas** — velocidades, ángulos y distancias por keypoint, que son la entrada real de Strike Candidate Detection.




### Probando la funcionalidad

In [7]:
from pathlib import Path

from ufc_tracker.detection.weights import project_root

# En un notebook no existe __file__; project_root() sube hasta pyproject.toml.
PROJECT_ROOT = project_root(Path.cwd())

VIDEO_PATH = (
    PROJECT_ROOT
    / "data/splits/normal_men"
    / "fiziev_bahamondes__rafael_fiziev_vs_ignacio_bahamondes__normal_men_round1.mp4"
)
SAVE_PATH = PROJECT_ROOT / "outputs/generic-pose-smoke/TEST-POSES"

if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f"No se encontró el video: {VIDEO_PATH}")

# tercer "peleador" en el mismo frame y el pipeline aborta.
test = run_pose_pipeline(
    VIDEO_PATH,
    SAVE_PATH,
    max_frames=1000,
    min_track_frames=50,
)

print(test)

PosePipelineResult(output_dir=WindowsPath('C:/Users/Cristian/Documents/UFC TRACKER/outputs/generic-pose-smoke/TEST-POSES'), tracking_path=WindowsPath('C:/Users/Cristian/Documents/UFC TRACKER/outputs/generic-pose-smoke/TEST-POSES/tracking.jsonl'), pose_path=WindowsPath('C:/Users/Cristian/Documents/UFC TRACKER/outputs/generic-pose-smoke/TEST-POSES/pose.jsonl'), preview_path=WindowsPath('C:/Users/Cristian/Documents/UFC TRACKER/outputs/generic-pose-smoke/TEST-POSES/pose_preview.mp4'), metrics_path=WindowsPath('C:/Users/Cristian/Documents/UFC TRACKER/outputs/generic-pose-smoke/TEST-POSES/pose_metrics.json'), metadata_path=WindowsPath('C:/Users/Cristian/Documents/UFC TRACKER/outputs/generic-pose-smoke/TEST-POSES/run_metadata.json'))
